# Prepare model table: drop identifier/leakage columns

Takes `training_table_avg10.csv` (one row per game, produced by `build_training_table.ipynb`) and strips it down to a model-ready feature set:

- **Identifier columns** (`GAME_ID`, `SEASON`, `GAME_DATE`, `HOME_TEAM_ID`, `HOME_TEAM_ABBREVIATION`, `AWAY_TEAM_ID`, `AWAY_TEAM_ABBREVIATION`) are dropped - they identify the game but carry no predictive signal a model should use.
- **`HOME_PTS`/`AWAY_PTS`** are dropped - they're the final score, i.e. the result of the game, not something known before tip-off.
- **`HOME_WIN`** is dropped - the target for this model is `HOME_DIFF` (home margin of victory as a continuous regression target), and `HOME_WIN` is just `sign(HOME_DIFF)`, so keeping it would leak the label into the features.

Everything else - the rolled trailing-average features, the `DIFF_*` columns, and rest-day columns - passes through unchanged, plus the `HOME_DIFF` label itself.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from data_import import settings

In [2]:
# Parameters - must match the N_GAMES used to produce the input training table.
N_GAMES = 10
INPUT_PATH = settings.PROCESSED_DATA_DIR / f"training_table_avg{N_GAMES}.csv"
OUTPUT_PATH = settings.PROCESSED_DATA_DIR / f"model_table_avg{N_GAMES}.csv"

## Load training table

In [3]:
games = pd.read_csv(INPUT_PATH)
print(f"loaded {len(games)} rows, {len(games.columns)} columns from {INPUT_PATH}")
games.shape

loaded 4293 rows, 120 columns from /Users/jacobgipson/NBA Betting Pipeline/NBA_Betting_Pipeline/data/processed/training_table_avg10.csv


(4293, 120)

## Drop identifier and leakage columns

- Identifiers: not predictive, just label/metadata for a given row.
- `HOME_PTS`/`AWAY_PTS`: the game's final score - only known *after* the game, so they can't be features.
- `HOME_WIN`: redundant with (and derivable from) the `HOME_DIFF` target - dropped to avoid leaking the label.

In [4]:
ID_COLS = [
    "GAME_ID", "SEASON", "GAME_DATE",
    "HOME_TEAM_ID", "HOME_TEAM_ABBREVIATION",
    "AWAY_TEAM_ID", "AWAY_TEAM_ABBREVIATION",
]
OUTCOME_LEAKAGE_COLS = ["HOME_PTS", "AWAY_PTS", "HOME_WIN"]
DROP_COLS = ID_COLS + OUTCOME_LEAKAGE_COLS

missing = [c for c in DROP_COLS if c not in games.columns]
assert not missing, f"expected column(s) not found in training table: {missing}"

TARGET_COL = "HOME_DIFF"
assert TARGET_COL in games.columns, f"target column {TARGET_COL} not found in training table"

model_table = games.drop(columns=DROP_COLS)
print(f"dropped {len(DROP_COLS)} columns: {DROP_COLS}")
model_table.shape

dropped 10 columns: ['GAME_ID', 'SEASON', 'GAME_DATE', 'HOME_TEAM_ID', 'HOME_TEAM_ABBREVIATION', 'AWAY_TEAM_ID', 'AWAY_TEAM_ABBREVIATION', 'HOME_PTS', 'AWAY_PTS', 'HOME_WIN']


(4293, 110)

## Sanity checks

In [5]:
assert not set(DROP_COLS) & set(model_table.columns), "a dropped column is still present"
print("OK: none of the dropped columns remain")

assert TARGET_COL in model_table.columns, f"{TARGET_COL} missing from model table"
print(f"OK: target column {TARGET_COL} is present")

nulls = model_table.isna().sum()
bad_nulls = nulls[nulls > 0]
assert bad_nulls.empty, f"unexpected nulls:\n{bad_nulls}"
print("OK: no nulls in any column")

model_table.head()

OK: none of the dropped columns remain
OK: target column HOME_DIFF is present
OK: no nulls in any column


,HOME_FGM_AVG10,HOME_FGA_AVG10,HOME_FG3M_AVG10,HOME_FG3A_AVG10,HOME_FTM_AVG10,HOME_FTA_AVG10,HOME_OREB_AVG10,HOME_DREB_AVG10,HOME_REB_AVG10,HOME_AST_AVG10,...,DIFF_TM_TOV_PCT_AVG10,DIFF_OFF_RATING_AVG10,DIFF_REB_PCT_AVG10,DIFF_DEF_RATING_AVG10,DIFF_NET_RATING_AVG10,HOME_REST,AWAY_REST,REST_DIFF,HOME_SPREAD,HOME_DIFF
0,40.8,91.6,11.4,31.9,17.0,23.6,11.7,34.3,46.0,26.5,...,0.010997,-2.020774,-0.016520,-3.138043,1.117268,1.0,0.0,1.0,-2.0,-8
1,40.1,85.1,9.5,30.5,20.3,25.9,10.1,36.1,46.2,20.6,...,-0.004471,2.924630,0.023257,-1.650133,4.574763,1.0,1.0,0.0,-5.0,-7
2,39.5,86.4,11.0,29.0,21.0,24.3,10.8,32.9,43.7,23.9,...,0.028599,-5.503159,-0.017682,1.662546,-7.165705,0.0,0.0,0.0,-2.0,14
3,40.2,92.3,10.6,32.2,16.1,22.9,12.0,33.4,45.4,26.6,...,-0.026497,-6.512851,-0.033344,2.087630,-8.600481,1.0,1.0,0.0,1.5,-10
4,41.2,90.8,15.3,40.4,19.4,24.2,11.3,33.4,44.7,28.4,...,0.000419,-0.244228,-0.009865,2.912236,-3.156464,1.0,1.0,0.0,5.0,-3


## Save model table

In [6]:
settings.PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
model_table.to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(model_table)} rows, {len(model_table.columns)} columns to {OUTPUT_PATH}")

wrote 4293 rows, 110 columns to /Users/jacobgipson/NBA Betting Pipeline/NBA_Betting_Pipeline/data/processed/model_table_avg10.csv
